In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
groq_api_key = os.getenv("GROQ_API_KEY")


In [4]:
from langchain_groq import ChatGroq
model = ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000016B78B65F90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000016B78B65EA0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [5]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content = " my name is srimad , i am in btech")])


AIMessage(content="Nice to meet you, Srimad. You're in your B.Tech program, I assume that's a 4-year undergraduate degree in Engineering. Which stream are you pursuing, and what's your favorite subject in engineering so far?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 47, 'total_tokens': 96, 'completion_time': 0.140974706, 'completion_tokens_details': None, 'prompt_time': 0.002559174, 'prompt_tokens_details': None, 'queue_time': 0.047573864, 'total_time': 0.14353388}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_1151d4f23c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c70e8-9c2a-7f71-93df-8a83c6d50150-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 49, 'total_tokens': 96})

In [6]:
from langchain_core.messages import AIMessage
model.invoke(
[
    HumanMessage(content= "my name is srimad , i am in btech"),
    AIMessage(content="Nice to meet you, Srimad. Being a B Tech student can be a challenging yet rewarding experience. Which branch of engineering are you studying? Are you in your first year or have you progressed to a higher semester? "),
    HumanMessage(content = "hey whats my name what do i do")
]

)

AIMessage(content="Your name is Srimad, and you're a B Tech student. What's your current field of study (e.g., CSE, ECE, ME, CE, etc.)?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 111, 'total_tokens': 151, 'completion_time': 0.087861061, 'completion_tokens_details': None, 'prompt_time': 0.008557075, 'prompt_tokens_details': None, 'queue_time': 0.046161975, 'total_time': 0.096418136}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_ff2b098aaf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c70e8-9f9e-7332-b609-ded9ea76f01a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 111, 'output_tokens': 40, 'total_tokens': 151})

In [7]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [8]:
config = {"configurable":{"session_id":"chat1"}}

In [9]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name srimad and i m in my 2nd year of btech")],
    config=config
)

In [10]:
response.content

"Nice to meet you, Srimad. How's life as a 2nd-year B.Tech student going so far? Are you enjoying the courses and projects? Which branch of engineering are you studying?"

In [11]:
#change the config
config1 = {"configurable": {"session_id": "chat2"}}

response = with_message_history.invoke(
    [HumanMessage(content="whats my name")],
    config=config1
)

print(response.content)


I don't have any information about your name. I'm a large language model, I don't have personal knowledge or memories of our previous conversations, so I don't know your name unless you've told me. If you want to share your name with me, I'd be happy to chat with you!


In [12]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is srimad")],
    config=config1
)
response.content

"Nice to meet you, Srimad. It's a unique and interesting name. Is there something specific you'd like to talk about or ask, or are you just looking to chat? I'm here to listen and help if I can."

In [13]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'Your name is Srimad.'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [14]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant , ans all my questions and help me "

        ),
        MessagesPlaceholder(variable_name="message"),
        
    ]
)
chain = prompt|model

In [18]:
chain.invoke({
    "message": [HumanMessage(content="my name is srimad")]
})



AIMessage(content="Nice to meet you, Srimad! I'm happy to be your assistant and help you with any questions or tasks you may have. What's on your mind today? Do you have a specific topic you'd like to discuss or a problem you need help with?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 54, 'total_tokens': 109, 'completion_time': 0.279418834, 'completion_tokens_details': None, 'prompt_time': 0.002688438, 'prompt_tokens_details': None, 'queue_time': 0.044728832, 'total_time': 0.282107272}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c70f7-e258-7851-8a24-4f87f9ca6a0c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 54, 'output_tokens': 55, 'total_tokens': 109})

In [ ]:
with_message_history= RunnableWithMessageHistory(chain,get_session_history)